# Reto proyecto Machine Learning II - Predicción del corte de diamantes

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

## 2. Carga del dataset

In [ ]:
df = pd.read_csv('/workspace/diamonds.csv')
df.head()

## 3. Exploración inicial

In [ ]:
print('Dimensiones:', df.shape)
print('\nTipos de datos:')
print(df.dtypes)
print('\nValores nulos:')
print(df.isnull().sum())
print('\nEstadísticas descriptivas:')
display(df.describe(include='all'))

## 4. Muestra para acelerar el entrenamiento

In [ ]:
df = df.sample(n=5000, random_state=42)
df.shape

## 5. Variables numéricas y categóricas

In [ ]:
target_col = 'cut'
categorical_cols = ['color', 'clarity']
numeric_cols = ['carat', 'depth', 'table', 'price', 'x', 'y', 'z']

print('Variables categóricas:', categorical_cols)
print('Variables numéricas:', numeric_cols)
print('Variable objetivo:', target_col)

## 6. Imputación de valores faltantes en columnas numéricas

In [ ]:
imputer = SimpleImputer(strategy='mean')
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

print(df[numeric_cols].isnull().sum())

## 7. Codificación de variables categóricas con LabelEncoder

In [ ]:
le_color = LabelEncoder()
le_clarity = LabelEncoder()
le_cut = LabelEncoder()

df['color'] = le_color.fit_transform(df['color'])
df['clarity'] = le_clarity.fit_transform(df['clarity'])
df['cut_encoded'] = le_cut.fit_transform(df['cut'])

df[['color', 'clarity', 'cut', 'cut_encoded']].head()

## 8. Separación de variables X e y

In [ ]:
X = df[['carat', 'color', 'clarity', 'depth', 'table', 'price', 'x', 'y', 'z']]
y = df['cut_encoded']

print('X shape:', X.shape)
print('y shape:', y.shape)

## 9. División en entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Entrenamiento:', X_train.shape, y_train.shape)
print('Prueba:', X_test.shape, y_test.shape)

## 10. Estandarización de variables

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled[:5]

## 11. Definición de modelos

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'SVC': SVC(probability=True, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42)
}

models

## 12. Entrenamiento y evaluación de los modelos

In [ ]:
results = []

for name, model in models.items():
    if name in ['KNN', 'SVC', 'Logistic Regression']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    results.append({
        'Modelo': name,
        'Accuracy': acc,
        'F1_weighted': f1
    })

    print('=' * 60)
    print(name)
    print('Accuracy:', acc)
    print('F1 weighted:', f1)
    print('\nMatriz de confusión:')
    print(confusion_matrix(y_test, y_pred))
    print('\nClassification report:')
    print(classification_report(y_test, y_pred, zero_division=0))

results_df = pd.DataFrame(results).sort_values(by='F1_weighted', ascending=False)
results_df

## 13. Comparación visual de accuracies

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=results_df, x='Modelo', y='Accuracy')
plt.title('Comparación de Accuracy por modelo')
plt.xticks(rotation=20)
plt.show()

## 14. Comparación visual de F1-score

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=results_df, x='Modelo', y='F1_weighted')
plt.title('Comparación de F1-score ponderado por modelo')
plt.xticks(rotation=20)
plt.show()

## 15. Mejor modelo

In [ ]:
best_model_name = results_df.iloc[0]['Modelo']
best_f1 = results_df.iloc[0]['F1_weighted']

print('Mejor modelo:', best_model_name)
print('Mejor F1-score:', best_f1)

print('\nEtiquetas originales de la variable objetivo:')
print(list(le_cut.classes_))

## 16. Conclusión

In [ ]:
print('Se cargó diamonds.csv con ruta absoluta.')
print('Se imputaron los valores faltantes numéricos con la media usando SimpleImputer.')
print('Se codificaron las variables categóricas con LabelEncoder.')
print('Se estandarizaron las variables numéricas con StandardScaler.')
print('Se entrenaron cinco modelos de clasificación y se compararon con accuracy, matriz de confusión y classification_report.')
print('El mejor modelo se eligió según el F1-score ponderado.')